# STV partial optimizer demo

This notebook demonstrates the consolidated numerical optimizer in
`stv_partial_optimizer.py`.

The public function is:

```python
result = spo.minimize_partial(base_point, theta, radius, quota)
```

It automatically selects between two PGD paths:

1. **`single_row_reduced`** when `theta` is supported on one row and preserves that row total;
2. **`general_recursive`** for every other nonzero direction.

Both paths optimize over the nonnegative $L^1$ diamond

$$
\{x\ge 0:\lVert x-x^0\rVert_1\le r\}.
$$

No condition $T_j\ge q$ is imposed during optimization.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

import numpy as np
import pandas as pd

# This works when the notebook and module are in the same directory.
module_dir = Path.cwd()
if not (module_dir / "stv_partial_optimizer.py").exists():
    module_dir = Path("/mnt/data")
sys.path.insert(0, str(module_dir))

import stv_partial_optimizer as spo

print("module version:", spo.__version__)

module version: 1.0.0


## Helpers for synthetic regular-domain examples

These Gamma-distributed basepoints are only benchmark inputs; they are not intended as election models.

In [2]:
def make_regular_basepoint(degree, total_mass, quota, rng):
    shape = (2**degree, 3)
    zero_theta = np.zeros(shape, dtype=float)

    for _ in range(2_000):
        raw = rng.gamma(shape=2.0, scale=1.0, size=shape)
        base = raw * (total_mass / raw.sum())
        tallies = spo.evaluate_directional_derivative(
            base, zero_theta, quota
        ).winner_tallies
        if np.all(tallies > quota):
            return base

    raise RuntimeError("Could not generate a regular-domain basepoint")


def same_row_theta(shape, row, plus_col=0, minus_col=1):
    theta = np.zeros(shape, dtype=float)
    theta[row, plus_col] = 1.0
    theta[row, minus_col] = -1.0
    return theta


def cross_row_theta(shape, rng):
    plus_row, minus_row = rng.choice(shape[0], size=2, replace=False)
    plus_col, minus_col = rng.integers(0, 3, size=2)
    theta = np.zeros(shape, dtype=float)
    theta[plus_row, plus_col] = 1.0
    theta[minus_row, minus_col] = -1.0
    return theta


rng = np.random.default_rng(20260619)

## Same-row direction: automatic reduced path

For a direction supported in one row with entries summing to zero,

$$
D_\theta M = (\theta_c-\theta_l)w_S(s).
$$

The optimizer therefore works in row-total coordinates and returns a point in the original `(row, c/l/o)` coordinates.

In [3]:
degree = 2
quota = 100.0
radius = 20.0
base = make_regular_basepoint(degree, total_mass=1_000.0, quota=quota, rng=rng)
theta = same_row_theta(base.shape, row=3, plus_col=0, minus_col=1)

reduced = spo.minimize_partial(base, theta, radius, quota)

pd.Series({
    "path": reduced.path,
    "minimum": reduced.minimum,
    "iterations": reduced.nit,
    "evaluations": reduced.nfev,
    "active_dimension": reduced.active_dimension,
    "L1 distance": reduced.l1_distance,
    "feasible": reduced.feasible,
    "success": reduced.success,
    "projected-gradient residual": reduced.projected_gradient_norm,
})

path                           single_row_reduced
minimum                                  1.220456
iterations                                     12
evaluations                                    12
active_dimension                                3
L1 distance                                  20.0
feasible                                     True
success                                      True
projected-gradient residual                   0.0
dtype: object

### Check the reduced answer against the general recursive path

Setting `specialize_single_row=False` forces the same problem through the general evaluator.

In [4]:
general_for_same_theta = spo.minimize_partial(
    base,
    theta,
    radius,
    quota,
    specialize_single_row=False,
)

pd.Series({
    "reduced path": reduced.path,
    "general path": general_for_same_theta.path,
    "reduced minimum": reduced.minimum,
    "general minimum": general_for_same_theta.minimum,
    "absolute difference": abs(
        reduced.minimum - general_for_same_theta.minimum
    ),
})

reduced path           single_row_reduced
general path            general_recursive
reduced minimum                  1.220456
general minimum                  1.220456
absolute difference                   0.0
dtype: object

## Cross-row direction: unified recursive path

With one `+1` and one `-1` in different rows, row totals change. The dispatcher therefore computes the directional derivative and its exact gradient recursively in the original coordinates.

In [5]:
theta_cross = cross_row_theta(base.shape, rng)
general = spo.minimize_partial(base, theta_cross, radius, quota)

pd.Series({
    "path": general.path,
    "minimum": general.minimum,
    "iterations": general.nit,
    "evaluations": general.nfev,
    "active_dimension": general.active_dimension,
    "L1 distance": general.l1_distance,
    "feasible": general.feasible,
    "success": general.success,
    "projected-gradient residual": general.projected_gradient_norm,
})

path                           general_recursive
minimum                                -1.454408
iterations                                    15
evaluations                                   15
active_dimension                              12
L1 distance                                 20.0
feasible                                    True
success                                     True
projected-gradient residual                  0.0
dtype: object

## Degree-5 timing benchmark

The earlier benchmark was approximately **55 ms per cross-row optimization** on its test run. Runtime is machine-dependent, so this cell measures the consolidated module directly rather than hard-coding that number.

The small and million-ballot problems are geometrically equivalent rescalings:

- small: total mass 1,000; quota 100; radius 20;
- large: total mass 1,000,000; quota 100,000; radius 20,000.

All benchmark directions have one `+1` and one `-1` in different rows, ensuring use of the general recursive path.

In [6]:
degree = 5
small_mass = 1_000.0
small_quota = 100.0
small_radius = 20.0
scale = 1_000.0
n_cases = 30

base_small = make_regular_basepoint(
    degree, small_mass, small_quota, rng
)
base_large = scale * base_small
benchmark_thetas = [
    cross_row_theta(base_small.shape, rng)
    for _ in range(n_cases)
]

records = []
for label, benchmark_base, benchmark_radius, benchmark_quota in (
    ("small", base_small, small_radius, small_quota),
    (
        "million_ballot_scale",
        base_large,
        scale * small_radius,
        scale * small_quota,
    ),
):
    for case, benchmark_theta in enumerate(benchmark_thetas):
        started = perf_counter()
        result = spo.minimize_partial(
            benchmark_base,
            benchmark_theta,
            benchmark_radius,
            benchmark_quota,
            n_starts=1,
        )
        elapsed = perf_counter() - started
        records.append({
            "scale": label,
            "case": case,
            "milliseconds": 1_000 * elapsed,
            "iterations": result.nit,
            "evaluations": result.nfev,
            "minimum": result.minimum,
            "residual": result.projected_gradient_norm,
            "feasible": result.feasible,
            "success": result.success,
            "path": result.path,
        })

benchmark = pd.DataFrame(records)
summary = benchmark.groupby("scale").agg(
    median_ms=("milliseconds", "median"),
    mean_ms=("milliseconds", "mean"),
    p90_ms=("milliseconds", lambda x: x.quantile(0.90)),
    median_iterations=("iterations", "median"),
    maximum_iterations=("iterations", "max"),
    median_evaluations=("evaluations", "median"),
    all_feasible=("feasible", "all"),
    all_converged=("success", "all"),
)
summary

,median_ms,mean_ms,p90_ms,median_iterations,maximum_iterations,median_evaluations,all_feasible,all_converged
scale,,,,,,,,
million_ballot_scale,34.233416,37.239326,45.695892,16.0,27,16.0,True,True
small,34.715255,38.361228,46.162055,16.0,27,16.0,True,True


In [7]:
small_median_ms = summary.loc["small", "median_ms"]
large_median_ms = summary.loc["million_ballot_scale", "median_ms"]

pd.Series({
    "1,000 small-scale optimizations, median-based seconds": (
        1_000 * small_median_ms / 1_000
    ),
    "1,000 million-scale optimizations, median-based seconds": (
        1_000 * large_median_ms / 1_000
    ),
    "large/small median runtime ratio": (
        large_median_ms / small_median_ms
    ),
})

1,000 small-scale optimizations, median-based seconds      34.715255
1,000 million-scale optimizations, median-based seconds    34.233416
large/small median runtime ratio                            0.986120
dtype: float64

## Optional single-row degree-5 benchmark

This demonstrates that the automatic reduced path is used for a same-row direction.

In [8]:
same_row_cases = []
for row in rng.integers(0, 2**degree, size=20):
    benchmark_theta = same_row_theta(
        base_large.shape,
        row=int(row),
        plus_col=0,
        minus_col=1,
    )
    started = perf_counter()
    result = spo.minimize_partial(
        base_large,
        benchmark_theta,
        scale * small_radius,
        scale * small_quota,
    )
    same_row_cases.append({
        "milliseconds": 1_000 * (perf_counter() - started),
        "iterations": result.nit,
        "active_dimension": result.active_dimension,
        "path": result.path,
    })

same_row_benchmark = pd.DataFrame(same_row_cases)
same_row_benchmark.agg({
    "milliseconds": ["median", "mean", "max"],
    "iterations": ["median", "mean", "max"],
    "active_dimension": ["median", "mean", "max"],
})

,milliseconds,iterations,active_dimension
median,20.239118,12.5,28.0
mean,20.512283,11.6,28.3
max,38.645768,16.0,31.0


## Cheap cache key for the online audit driver

For the expected two-entry directions, the ordered pair of flattened indices uniquely identifies `theta` at a fixed vertex.

In [9]:
key = spo.two_entry_theta_key(theta_cross)
key

(7, 9)

In [10]:
theta_cross

array([[ 0.,  0.,  0.],
       [ 0.,  0.,  0.],
       [ 0.,  1.,  0.],
       [-1.,  0.,  0.]])

## Production call

```python
key = spo.two_entry_theta_key(theta)

if key not in vertex.partial_bounds:
    result = spo.minimize_partial(
        vertex.base_point,
        theta,
        radius=vertex.radius,
        quota=vertex.quota,
        n_starts=1,
    )
    vertex.partial_bounds[key] = result.minimum

minimum = vertex.partial_bounds[key]
```

`result.minimum` is the numerical PGD candidate. As discussed, it is not yet a mathematically certified lower bound on the true minimum unless paired with a global-optimality or curvature argument.